# CardioIA - Fase 2 | Parte 1
## Extração de sintomas e sugestão de diagnóstico

Neste notebook a gente lê as frases de pacientes do arquivo `frases_sintomas.txt`, procura os sintomas que estão no mapa de conhecimento (`mapa_conhecimento.csv`) e sugere o diagnóstico mais provável para cada relato.

A ideia é simples: cada sintoma encontrado na frase soma um ponto para as doenças ligadas a ele. Quando os dois sintomas de uma mesma linha do mapa aparecem juntos, a doença ganha um ponto extra, porque a combinação é mais forte do que um sintoma isolado.

In [1]:
import re
import unicodedata
import pandas as pd

### Normalização do texto
Pacientes escrevem de jeitos diferentes (com acento, sem acento, maiúscula...). Antes de comparar, deixamos tudo em minúsculo e sem acentos.

In [2]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", texto).strip()

normalizar("Dor no PEITO e aperto no Tórax")

'dor no peito e aperto no torax'

### Leitura dos arquivos

In [3]:
with open("frases_sintomas.txt", encoding="utf-8") as arquivo:
    frases = [linha.strip() for linha in arquivo if linha.strip()]

print(f"{len(frases)} frases carregadas\n")
for i, frase in enumerate(frases, start=1):
    print(f"{i:>2}. {frase}")

10 frases carregadas

 1. Há dois dias estou com uma dor no peito em aperto que vai para o braço esquerdo e piora quando faço esforço físico, hoje não consegui nem subir a escada de casa.
 2. Sinto cansaço constante há uma semana, mesmo depois de descansar, minhas pernas ficam inchadas no fim do dia e preciso dormir com dois travesseiros para conseguir respirar.
 3. Desde ontem sinto o coração disparado e umas palpitações do nada, fiquei tonto no trabalho e tive que sentar até passar.
 4. Há três semanas sinto uma dor no peito quando caminho rápido, mas ela passa quando paro para descansar, por isso parei de ir a pé para o mercado.
 5. Faz uns dez dias que acordo com dor de cabeça na nuca e a visão fica embaçada, não consigo me concentrar no serviço e meu rosto fica vermelho.
 6. Hoje de manhã meu braço direito ficou dormente, a boca ficou torta e comecei a falar enrolado, minha esposa precisou me ajudar a levantar da cama.
 7. Há quatro dias estou com uma dor no peito que piora quando

In [4]:
mapa = pd.read_csv("mapa_conhecimento.csv")
print(f"{len(mapa)} associações no mapa de conhecimento")
mapa.head(10)

64 associações no mapa de conhecimento


,sintoma_1,sintoma_2,doenca_associada
0,dor no peito,aperto no peito,Infarto Agudo do Miocárdio
1,dor no peito em aperto,braço esquerdo,Infarto Agudo do Miocárdio
2,aperto no tórax,suor frio,Infarto Agudo do Miocárdio
3,pressão no peito,dor no braço,Infarto Agudo do Miocárdio
4,dor que irradia,mandíbula,Infarto Agudo do Miocárdio
5,dor forte no peito,náusea,Infarto Agudo do Miocárdio
6,queimação no peito,suor frio,Infarto Agudo do Miocárdio
7,dor no peito,piora quando faço esforço,Infarto Agudo do Miocárdio
8,dor no peito,esforço físico,Infarto Agudo do Miocárdio
9,dor no peito,falta de ar,Infarto Agudo do Miocárdio


In [5]:
mapa["doenca_associada"].value_counts()

doenca_associada
Infarto Agudo do Miocárdio                     10
Angina                                          7
Insuficiência Cardíaca                          7
Arritmia                                        6
Acidente Vascular Cerebral (AVC)                6
Hipertensão Arterial                            5
Endocardite Infecciosa                          5
Trombose Venosa Profunda / Embolia Pulmonar     5
Miocardite                                      4
Pericardite                                     4
Doença Arterial Periférica                      3
Estenose Aórtica                                2
Name: count, dtype: int64

### Montando a base de conhecimento
Para cada doença guardamos o conjunto de expressões ligadas a ela e também os pares (sintoma 1 + sintoma 2) de cada linha.

In [6]:
sintomas_por_doenca = {}
pares_por_doenca = {}

for _, linha in mapa.iterrows():
    doenca = linha["doenca_associada"]
    s1 = normalizar(linha["sintoma_1"])
    s2 = normalizar(linha["sintoma_2"])
    sintomas_por_doenca.setdefault(doenca, set()).update([s1, s2])
    pares_por_doenca.setdefault(doenca, []).append((s1, s2))

for doenca, sintomas in sintomas_por_doenca.items():
    print(f"{doenca}: {len(sintomas)} expressões")

Infarto Agudo do Miocárdio: 16 expressões
Angina: 11 expressões
Insuficiência Cardíaca: 14 expressões
Arritmia: 12 expressões
Hipertensão Arterial: 10 expressões
Acidente Vascular Cerebral (AVC): 12 expressões
Pericardite: 7 expressões
Miocardite: 7 expressões
Trombose Venosa Profunda / Embolia Pulmonar: 10 expressões
Endocardite Infecciosa: 10 expressões
Estenose Aórtica: 4 expressões
Doença Arterial Periférica: 6 expressões


### Extraindo quando o sintoma começou
Além dos sintomas, as frases trazem a informação de tempo ("há dois dias", "desde ontem"...). Uma regex simples já resolve boa parte dos casos.

In [7]:
padroes_tempo = [
    r"\b(?:ha|faz)\s+(?:uns |umas |cerca de |mais de )?\w+\s+(?:horas?|dias?|semanas?|mes|meses|anos?)\b",
    r"\bdesde ontem\b",
    r"\bdesde \w+\b",
    r"\bhoje de manha\b",
    r"\bhoje\b",
    r"\bontem\b",
]

def extrair_tempo(frase):
    texto = normalizar(frase)
    for padrao in padroes_tempo:
        achou = re.search(padrao, texto)
        if achou:
            return achou.group(0)
    return "não informado"

extrair_tempo("Faz uns dez dias que acordo com dor de cabeça")

'faz uns dez dias'

### Função de diagnóstico

In [8]:
def diagnosticar(frase):
    texto = normalizar(frase)
    pontos = {}
    encontrados = {}

    for doenca, sintomas in sintomas_por_doenca.items():
        achados = sorted(s for s in sintomas if s in texto)
        if not achados:
            continue
        bonus = sum(1 for s1, s2 in pares_por_doenca[doenca] if s1 in texto and s2 in texto)
        pontos[doenca] = len(achados) + bonus
        encontrados[doenca] = achados

    ranking = sorted(pontos.items(), key=lambda item: item[1], reverse=True)
    sintomas_frase = sorted({s for lista in encontrados.values() for s in lista})
    return ranking, sintomas_frase

### Rodando para os 10 pacientes

In [9]:
resultados = []

for i, frase in enumerate(frases, start=1):
    ranking, sintomas = diagnosticar(frase)
    tempo = extrair_tempo(frase)

    print(f"Paciente {i}")
    print(f"  Relato: {frase}")
    print(f"  Início dos sintomas: {tempo}")
    print(f"  Sintomas identificados: {', '.join(sintomas) if sintomas else 'nenhum'}")

    if ranking:
        principal, pontuacao = ranking[0]
        outras = [f"{d} ({p})" for d, p in ranking[1:3]]
        print(f"  Diagnóstico sugerido: {principal} (pontuação {pontuacao})")
        if outras:
            print(f"  Outras possibilidades: {', '.join(outras)}")
    else:
        principal, pontuacao, outras = "Sem sugestão", 0, []
        print("  Diagnóstico sugerido: nenhum sintoma do mapa foi encontrado")
    print()

    resultados.append({
        "paciente": i,
        "frase": frase,
        "inicio_sintomas": tempo,
        "sintomas_identificados": "; ".join(sintomas),
        "diagnostico_sugerido": principal,
        "pontuacao": pontuacao,
        "outras_possibilidades": "; ".join(outras),
    })

Paciente 1
  Relato: Há dois dias estou com uma dor no peito em aperto que vai para o braço esquerdo e piora quando faço esforço físico, hoje não consegui nem subir a escada de casa.
  Início dos sintomas: ha dois dias
  Sintomas identificados: braco esquerdo, dor no peito, dor no peito em aperto, esforco fisico, piora quando faco esforco
  Diagnóstico sugerido: Infarto Agudo do Miocárdio (pontuação 8)
  Outras possibilidades: Angina (1), Pericardite (1)

Paciente 2
  Relato: Sinto cansaço constante há uma semana, mesmo depois de descansar, minhas pernas ficam inchadas no fim do dia e preciso dormir com dois travesseiros para conseguir respirar.
  Início dos sintomas: ha uma semana
  Sintomas identificados: cansaco, cansaco constante, dois travesseiros, inchadas no fim do dia, mesmo depois de descansar
  Diagnóstico sugerido: Insuficiência Cardíaca (pontuação 6)

Paciente 3
  Relato: Desde ontem sinto o coração disparado e umas palpitações do nada, fiquei tonto no trabalho e tive que s

In [10]:
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("resultado_diagnosticos.csv", index=False, encoding="utf-8")
df_resultados[["paciente", "inicio_sintomas", "diagnostico_sugerido", "pontuacao", "outras_possibilidades"]]

,paciente,inicio_sintomas,diagnostico_sugerido,pontuacao,outras_possibilidades
0,1,ha dois dias,Infarto Agudo do Miocárdio,8,Angina (1); Pericardite (1)
1,2,ha uma semana,Insuficiência Cardíaca,6,
2,3,desde ontem,Arritmia,5,
3,4,ha tres semanas,Angina,7,Infarto Agudo do Miocárdio (1); Pericardite (1)
4,5,faz uns dez dias,Hipertensão Arterial,6,
5,6,hoje de manha,Acidente Vascular Cerebral (AVC),4,
6,7,ha quatro dias,Pericardite,4,Infarto Agudo do Miocárdio (1); Angina (1)
7,8,ha duas semanas,Miocardite,5,Infarto Agudo do Miocárdio (1); Arritmia (1)
8,9,ha cinco dias,Trombose Venosa Profunda / Embolia Pulmonar,6,Infarto Agudo do Miocárdio (1); Miocardite (1)
9,10,ha um mes,Endocardite Infecciosa,10,Pericardite (1)


### Testando com um relato novo

In [11]:
novo = "Desde ontem sinto um aperto no tórax com suor frio e náusea, não consegui ir trabalhar"
ranking, sintomas = diagnosticar(novo)
print("Sintomas:", sintomas)
print("Início:", extrair_tempo(novo))
print("Ranking:", ranking)

Sintomas: ['aperto no torax', 'nausea', 'suor frio']
Início: desde ontem
Ranking: [('Infarto Agudo do Miocárdio', 4)]


### Observações e limitações

- O sistema funciona por correspondência de texto, então só reconhece sintomas escritos de um jeito parecido com o que está no mapa. Quanto mais variações de escrita o CSV tiver, mais casos ele cobre.
- Sintomas genéricos como "dor no peito", "febre" e "falta de ar" aparecem em várias doenças. Por isso usamos a pontuação e mostramos também as outras possibilidades, e não só uma resposta.
- Negação não é tratada: uma frase como "não sinto dor no peito" ainda contaria o sintoma. Seria um próximo passo de melhoria.
- O mapa foi montado pelo grupo a partir de material de estudo, não por um médico. O resultado é uma **sugestão de apoio**, nunca um diagnóstico final.